# Predicting Customer Churn in a Telecommunications Company
## A Business Intelligence & Data Mining Project

---

**Business Context:** Customer churn — when subscribers cancel their service — is one of the most critical challenges in telecommunications. Research consistently shows that retaining existing customers is significantly more cost-effective than acquiring new ones, making churn prediction a high-value business intelligence problem (Sikri et al., 2024). The model developed in this project is intended to support the Marketing and CRM teams in executing targeted retention campaigns by identifying high-risk customers, enabling the organisation to allocate resources efficiently and reduce revenue loss.

**Objective:** Apply the full BI and data mining lifecycle to build a predictive model that identifies at-risk customers and translates findings into actionable retention strategies.

**Dataset:** The publicly available IBM Telco Customer Churn dataset (IBM, 2018), containing 7,043 customer records with 21 features covering demographics, account information, and service subscriptions.

---
# Stage 1: Problem Definition & Data Scoping

## 1.1 Business Problem

TelcoPlus, a fictional telecommunications provider, faces an escalating churn rate that erodes its subscriber base. The VP of Customer Retention has commissioned a data-driven investigation to answer: *"Which customers are most likely to churn, and what are the key drivers behind their decision to leave?"*

This problem is strategically significant because customer retention is far more cost-effective than acquisition, with studies estimating acquisition costs at five to twenty-five times higher than retention costs (Sikri et al., 2024). In the telecommunications sector specifically, annual churn rates frequently exceed 30%, making it one of the most affected industries (Chang et al., 2024). A predictive churn model directly enables targeted offers, personalised outreach, and service improvements — shifting the retention strategy from reactive to proactive.

The intended users of this model are the Marketing and CRM teams, who would use individual customer churn risk scores to prioritise retention campaigns, allocate discount budgets, and design segment-specific interventions. The key decision the model supports is: *which customers should receive retention offers, and what type of offer would be most effective?*

## 1.2 Project Objectives

1. Explore the dataset to identify patterns and early indicators of churn.
2. Build and compare multiple classification models to predict customer churn.
3. Interpret model outputs to identify the most important churn drivers.
4. Apply customer segmentation to identify distinct behavioural profiles.
5. Recommend concrete, data-backed retention strategies with clear business justification.

## 1.3 Dataset Justification

The Telco Customer Churn dataset is well-suited because it represents a realistic scenario with a binary target variable (Churn: Yes/No), a mix of data types requiring meaningful pre-processing, and a class imbalance (~26–29% churn) that mirrors real-world conditions. It has been widely used in peer-reviewed churn prediction research as a benchmark dataset (Sikri et al., 2024; Sana et al., 2022; Chang et al., 2024).

## 1.4 Data Loading & Initial Inspection

In [4]:
# ─── Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
print("All libraries loaded successfully.")

All libraries loaded successfully.


In [5]:
# ─── Load the IBM Telco Customer Churn dataset ────────────
# Source: IBM Sample Data Sets (IBM, 2018), available via Kaggle and GitHub.
# Hosted in the project repository for reproducibility.

url = 'https://raw.githubusercontent.com/K-H-SOE/DataMining/main/WA_Fn-UseC_-Telco-Customer-Churn.csv'

try:
    df = pd.read_csv(url)
    print("Dataset loaded from project GitHub repository.")
except Exception:
    # Fallback: load from local file if no internet access
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
    print("Dataset loaded from local CSV file.")

print(f"Dataset shape: {df.shape}")
print(f"Churn rate: {df['Churn'].value_counts(normalize=True)['Yes']:.1%}")
df.head()

Dataset loaded from local CSV file.
Dataset shape: (7043, 21)
Churn rate: 26.5%


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Dataset Description

The dataset is the **IBM Telco Customer Churn** dataset (IBM, 2018), a publicly available real-world dataset originally published by IBM as a sample data resource and widely hosted on Kaggle. It contains **7,043 customer records** and **21 features** covering three categories:

- **Demographics:** gender, SeniorCitizen, Partner, Dependents
- **Account information:** tenure, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges
- **Services subscribed:** PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies

The binary target variable `Churn` (Yes/No) indicates whether the customer left the provider in the last month. The dataset exhibits a class imbalance of approximately 73% retained vs 27% churned, which mirrors real-world churn conditions.

**Limitations of the dataset:**
- It represents a **static snapshot** — customer behaviour is captured at a single point in time, whereas real churn risk evolves dynamically.
- It **lacks behavioural variables** such as call quality metrics, customer service complaint history, network outage exposure, and competitor pricing data that could improve predictive accuracy.
- **No timestamps** are provided, so seasonal or temporal patterns in churn cannot be analysed.
- The dataset is relatively **small** (7,043 rows), which limits the complexity of models that can be trained without overfitting.

In [ ]:
# ─── Data overview ──────────────────────────────────────────
print("Data Types:\n"); print(df.dtypes)
print(f"\nMissing/blank values: TotalCharges = {df['TotalCharges'].isin([' ']).sum()}")

---
# Stage 2: Exploratory Data Analysis & Pre-processing

With the business problem defined, we now explore, clean, and prepare the data for analysis. This stage is critical for understanding the data's structure, identifying quality issues, and uncovering initial patterns that will guide model selection. Effective data visualisation is a key business intelligence skill for communicating complex data clearly to stakeholders (Chang et al., 2024).

## 2.1 Data Cleaning

The `TotalCharges` column contains 11 blank entries stored as whitespace strings. These correspond to customers with zero tenure (brand-new subscribers) and are imputed with 0. The target variable is encoded as a binary integer for modelling purposes.